# Key-generation WOTS_PRF trace capture (SHA2-SLH-DSA)

This notebook records **individual** address-based WOTS_PRF invocations in the
logical order used by FIPS 205 `slh_keygen_internal -> xmss_node(top layer,
tree 0) -> wots_PKgen`. It is **not** a continuous hardware trace of an entire
key-generation operation. The host constructs each public address; the existing
PRF-only SimpleSerial firmware executes one PRF per waveform.

The WOTS hash chains and XMSS node hashing are not measured by this workflow.
The public experimental seed material is deterministic and **insecure**; it is
not the original private experiment's seed material.

`RUN_CAPTURE` is `False` by default. First inspect the plan without hardware.


## 1. Imports

Run Jupyter from the repository root or the `notebooks/` folder. Keep the
companion Python modules in `src/`.


In [ ]:
from __future__ import annotations
import os
import sys
from pathlib import Path

WORKDIR = Path.cwd().resolve()
PROJECT_ROOT = WORKDIR if (WORKDIR / "src").is_dir() else WORKDIR.parent
MODULE_DIR = PROJECT_ROOT / "src"
if not MODULE_DIR.is_dir():
    raise FileNotFoundError(f"Cannot find the repository src/ directory: {MODULE_DIR}")
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import keygen_wots_sequence_capture as keycap
import prf_capture_core as live

print("Capture module:", keycap.__file__)
print("Capture version:", keycap.VERSION)


## 2. Configuration

Start with the small 128f/Os example and increase counts only after verifying
your hardware, storage capacity and acquisition settings.

Set `SLH_DSA_FIRMWARE_DIR` to the firmware **copy inside the ChipWhisperer
build tree**, not to the copy in this repository. `SLH_DSA_RESULTS_DIR` is
optional (the default is the ignored `results/` folder). To reuse a private
signing-capture seed pool, set `SLH_DSA_SIGNING_SEED_SOURCE_DIR` to the existing
private result root. Do not commit experimental seed manifests or measured traces.


In [ ]:
# Expand to ("128f", "128s", "192f", "192s", "256f", "256s") for all sets.
PARAM_SETS = ("128f",)
COMPILER_OPTS = ("Os",)
NUM_SECRET_SEEDS = 1
SECRET_SEED_MASTER = int(os.environ.get("SLH_DSA_SECRET_SEED_MASTER", "424242"))

_firmware = os.environ.get("SLH_DSA_FIRMWARE_DIR", "").strip()
FIRMWARE_DIR = Path(_firmware).expanduser() if _firmware else None
RESULTS_BASE_DIR = Path(os.environ.get("SLH_DSA_RESULTS_DIR", str(PROJECT_ROOT / "results"))).expanduser()
_private_seed_source = os.environ.get("SLH_DSA_SIGNING_SEED_SOURCE_DIR", "").strip()
SHARED_SEED_SOURCE_BASE = Path(_private_seed_source).expanduser() if _private_seed_source else None

PLATFORM = "CW308_STM32F3"
CRYPTO_TARGET = "MBEDTLS"
GAIN_DB = 15.0
MAX_SAMPLES = 131124
NUMBER_OF_POINTS = -1  # -1 = automatic trigger-based calibration
TIMEOUT = 250
RESUME = True
VERIFY_ALL_PRF_OUTPUTS = False
FLUSH_EVERY_LEAVES = 1
RUN_CAPTURE = False  # Enable only with compatible hardware connected.


## 3. Preview without hardware

The top-layer XMSS key-generation PRF sequence contains
`2^h_prime * wots_len` WOTS_PRF invocations per secret seed. The size preview
uses `MAX_SAMPLES` for a conservative bound on raw float32 trace storage.


In [ ]:
plan = keycap.preview_keygen_plan(
    param_sets=PARAM_SETS,
    compiler_opts=COMPILER_OPTS,
    num_secret_seeds=NUM_SECRET_SEEDS,
    samples=MAX_SAMPLES,
)
for item in plan:
    print(f"{item['parameter_set']}/{item['compiler_opt']}: "
          f"{item['wots_keypairs']:,} keypairs x {item['wots_len']} chains "
          f"= {item['wots_prf_calls']:,} PRFs/seed; "
          f"{item['total_prf_calls']:,} PRFs for all seeds; "
          f"up to {item['estimated_trace_gib']:.3f} GiB raw traces")


## 4. Hardware acquisition

When enabled, the batch builds and programs the PRF-only target for each
parameter/optimization pair, calibrates ADC samples and captures isolated PRFs
in ascending keypair, then ascending WOTS chain order.

Completed captures are reused. Partial captures resume at committed WOTS
keypair boundaries. Use a new empty output directory for new configurations.


In [ ]:
if not RUN_CAPTURE:
    print("Preview only. Set RUN_CAPTURE=True after configuring your ChipWhisperer hardware.")
else:
    if FIRMWARE_DIR is None or not FIRMWARE_DIR.is_dir():
        raise FileNotFoundError("Set SLH_DSA_FIRMWARE_DIR to the ChipWhisperer firmware build directory")
    if not (FIRMWARE_DIR / "Makefile").is_file():
        raise FileNotFoundError(f"Makefile not found: {FIRMWARE_DIR}")
    import chipwhisperer as cw
    scope = cw.scope()
    target = None
    try:
        scope.default_setup()
        scope.gain.db = GAIN_DB
        scope.adc.samples = MAX_SAMPLES
        scope.adc.presamples = 0
        target = cw.target(scope, cw.targets.SimpleSerial)
        report = keycap.run_keygen_capture_batch(
            scope, target,
            firmware_dir=FIRMWARE_DIR,
            results_base_dir=RESULTS_BASE_DIR,
            param_sets=PARAM_SETS,
            compiler_opts=COMPILER_OPTS,
            num_secret_seeds=NUM_SECRET_SEEDS,
            secret_seed_master=SECRET_SEED_MASTER,
            shared_seed_source_base=SHARED_SEED_SOURCE_BASE,
            platform=PLATFORM,
            crypto_target=CRYPTO_TARGET,
            max_samples=MAX_SAMPLES,
            number_of_points=NUMBER_OF_POINTS,
            auto_calibrate_points=(NUMBER_OF_POINTS < 1),
            timeout=TIMEOUT,
            verify_first_n=(-1 if VERIFY_ALL_PRF_OUTPUTS else 5),
            flush_every_leaves=FLUSH_EVERY_LEAVES,
            resume=RESUME,
        )
        print("Completed conditions:", report["conditions"])
    finally:
        if target is not None:
            try:
                target.dis()
            except Exception:
                pass
        scope.dis()


## 5. Output format

```text
results/SLH_DSA_128f_sha2_keygen_prf_sequence/
  shared_secret_seeds.json             # Private; do not commit
  Os/seed_000/
    keygen_capture_manifest.json
    capture_traces_random.npy          # One isolated PRF trace per row
    capture_addrs_random.npy           # Associated 32-byte ADRS per row
    capture_params_random.npy
    capture_meta_columns.npy
    capture_capture_config.json
    capture_keygen_sequence.json
```

The `random` suffix follows the existing storage convention; the key-generation
addresses here are deterministic, **not independently randomized**.
